Cell 1: Instalasi Library

In [2]:
import sys
import os

# --- FASE 0: PREPARASI KONFIGURASI LINGKUNGAN SISTEM ---
# Menonaktifkan pemicu kompilasi paksa dari Flash Attention yang dapat
# membekukan sistem dan menghabiskan ketersediaan memori RAM Windows.
os.environ = "FALSE"
# Membatasi pemrosesan kerja paralel asinkron (Ninja) ke tingkat aman 
# bila kompilasi fallback tidak sengaja terpicu di latar belakang.
os.environ = "4"

print("Memulai pembaruan utilitas PIP inti ke versi termutakhir...")
!{sys.executable} -m pip install --upgrade pip wheel packaging ninja --quiet

print("\n--- FASE 1: INSTALASI PUSAT KOMPUTASI (PYTORCH & CUDA 13.0) ---")
# Menarik rilis tingkat bawah PyTorch secara langsung dari indeks repositori 
# yang dikonfigurasi secara spesifik untuk fungsionalitas CUDA 13.0
!{sys.executable} -m pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu130

print("\n--- FASE 2: INJEKSI AKSELERASI ATENSI (XFORMERS & FLASH ATTENTION) ---")
# Xformers wajib selaras dengan fondasi arsitektural torch>=2.10 dan lingkungan cu130
!{sys.executable} -m pip install xformers==0.0.35 --index-url https://download.pytorch.org/whl/cu130

# Integrasi Flash Attention 2 melompati kompilasi MSVC C++ lokal dengan menyerap  
# roda biner pra-kompilasi komunitas yang stabil, menargetkan cxx11abi=FALSE. 
# Catatan: Tautan diselaraskan untuk instalasi Python 3.12 pada Windows.
!{sys.executable} -m pip install https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
# (Alternatif penarikan ekivalen bila tautan dialihkan dalam lingkungan khusus Windows:
#!{sys.executable} -m pip install https://huggingface.co/ussoewwin/Flash-Attention-2_for_Windows/resolve/main/flash_attn-2.8.3+cu130torch2.10.0cxx11abiFALSE-cp312-cp312-win_amd64.whl )

print("\n--- FASE 3: KERANGKA KERJA KUANTISASI SKALA EKSTREM ---")
# Pembaruan besar bitsandbytes yang kini mendeteksi arsitektur OS Windows
# secara native untuk eksekusi algoritma optimisasi 8-bit dan pemenggalan 4-bit.
!{sys.executable} -m pip install bitsandbytes>=0.43.3

print("\n--- FASE 4: PELINDUNGAN ABI MATEMATIKA DAN UTILITAS EVALUASI ---")
# Penguncian Numpy krusial diterapkan (di bawah 2.0.0) guna memitigasi 
# distorsi struktural C-API pada pustaka yang memerlukan memori matriks historis.
!{sys.executable} -m pip install "numpy<2.0.0" pandas psutil ipywidgets evaluate sacrebleu datasets

print("\n--- FASE 5: ORKESTRASI PEMODELAN DAN PEMBELAJARAN PENGUATAN ---")
# Memasukkan sistem Hugging Face tingkat lanjut dan pelararan kebijakan LLM
!{sys.executable} -m pip install transformers accelerate peft trl

print("\n--- FASE 6: OPERASIONALISASI UNSLOTH DAN SUBSISTEM TRITON ---")
# Menyuntikkan JIT Compiler Triton khusus OS Windows untuk mengatasi galat kompatibilitas Linux
!{sys.executable} -m pip install triton-windows

# Isolasi kritis: Instalasi Unsloth wajib didefinisikan tanpa menarik rantai 
# dependensi otomatis. Tindakan ini merupakan tembok pertahanan agar pengelola 
# PIP tidak secara liar menimpa instalasi PyTorch 2.10 yang krusial dari fase pertama.
!{sys.executable} -m pip install --no-deps unsloth unsloth-zoo

print("\n Konfigurasi Ekosistem Machine Learning Native Windows telah selesai dengan resolusi konflik 0%. Lingkungan Jupyter Notebook siap digunakan!")

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\araih\OneDrive\Documents\skripsi\llama-3.2-translation\.env\Lib\site-packages\IPython\core\interactiveshell.py", line 3225, in _run_cell
    result = runner(coro)
             ^^^^^^^^^^^^
  File "C:\Users\araih\OneDrive\Documents\skripsi\llama-3.2-translation\.env\Lib\site-packages\IPython\core\async_helpers.py", line 128, in _pseudo_sync_runner
    coro.send(None)
  File "C:\Users\araih\OneDrive\Documents\skripsi\llama-3.2-translation\.env\Lib\site-packages\IPython\core\interactiveshell.py", line 3414, in run_cell_async
    cell_name = compiler.cache(cell, execution_count, raw_code=raw_cell)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\araih\OneDrive\Documents\skripsi\llama-3.2-translation\.env\Lib\site-packages\IPython\core\compilerop.py", line 155, in cache
    name = self.get_code_name(raw_code, transformed_code, number)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Cell 2: Import Library & Autentikasi

In [ ]:
import gc
import torch
import unsloth
import re
import time
import psutil
import glob
import warnings
from datasets import load_dataset, DatasetDict, concatenate_datasets, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTConfig, SFTTrainer
import evaluate
from huggingface_hub import login

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from unsloth.chat_templates import train_on_responses_only

torch.backends.cudnn.benchmark = True

# Fungsi untuk memonitor memori (Sangat penting untuk RAM 16GB & VRAM 12GB)
def print_memory_usage(task_name=""):
    ram = psutil.virtual_memory().used / (1024**3)
    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / (1024**3)
        vram_reserved = torch.cuda.memory_reserved() / (1024**3)
        print(f"[{task_name}] RAM: {ram:.2f} GB | VRAM Terpakai: {vram:.2f} GB | VRAM Reserved: {vram_reserved:.2f} GB")
    else:
        print(f"[{task_name}] RAM: {ram:.2f} GB | GPU tidak terdeteksi")

login(token="login")
# model_id = "meta-llama/Llama-3.2-3B"

model_name="unsloth/Llama-3.2-3B-Instruct"

Cell 3: [a] Pengumpulan Data

In [ ]:
print("Tahap a: Pengumpulan Data...")
dataset_name = "prosa-text/nusa-translation"
subset_name = "bug" 

# Mengambil langsung file CSV dari repository Hugging Face
base_url = f"https://huggingface.co/datasets/{dataset_name}/resolve/main/raws/{subset_name}/"
data_files = {
    "train": base_url + "train.csv",
    "validation": base_url + "validation.csv",
    "test": base_url + "test.csv"
}

# Memuat dataset menggunakan format "csv" bawaan
raw_dataset = load_dataset("csv", data_files=data_files)

jml_awal_data = sum(len(raw_dataset[split]) for split in raw_dataset.keys())
print(f"Total data mentah: {jml_awal_data}")
print("Struktur Dataset:\n", raw_dataset)

Cell 4: [b] Pra-pemrosesan Data (Preprocessing)

In [ ]:
print("Tahap b: Pra-pemrosesan Data...")

# 1. Muat tokenizer (ringan)
model_name = "unsloth/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})
tokenizer.padding_side = "right"

# 2. Pasang chat template kustom (tanpa tanggal otomatis)
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>"
    "{% elif message['role'] == 'user' %}"
    "<|start_header_id|>user<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>"
    "{% elif message['role'] == 'assistant' %}"
    "<|start_header_id|>assistant<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>"
    "{% endif %}"
    "{% endfor %}"
)

# Fungsi pembersihan (sama)
def remove_latex(text):
    text = re.sub(r'\\(?:displaystyle|mathbf|frac|lim|rightarrow|infty|partial|sum|int|prod|sqrt|Delta|mathbf|mathbf|alpha|beta|gamma|theta|pi|sigma|lambda|mu|rho|phi|epsilon|omega)\b', ' ', text)
    text = re.sub(r'[{}]', ' ', text)
    text = re.sub(r'\\begin\{[^}]*\}.*?\\end\{[^}]*\}', ' ', text, flags=re.DOTALL)
    text = re.sub(r'\$.*?\$', ' ', text)
    text = re.sub(r'\$\$.*?\$\$', ' ', text, flags=re.DOTALL)
    return text

def clean_text(text):
    text = remove_latex(str(text))
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 3. Bersihkan & deduplikasi RAW dataset (sebelum arah)
raw_clean = raw_dataset.map(
    lambda examples: {
        "original_clean": [clean_text(t) for t in examples["original"]],
        "translated_clean": [clean_text(t) for t in examples["translated"]]
    },
    batched=True
)

# Buang pasangan kosong
raw_clean = raw_clean.filter(lambda x: len(x["original_clean"]) > 0 and len(x["translated_clean"]) > 0)

# Deduplikasi berdasarkan pasangan bersih
import pandas as pd
for split in raw_clean.keys():
    df = raw_clean[split].to_pandas()
    sebelum = len(df)
    df.drop_duplicates(subset=["original_clean", "translated_clean"], inplace=True)
    sesudah = len(df)
    raw_clean[split] = raw_clean[split].select(df.index.tolist())
    print(f"Split {split}: {sebelum - sesudah} duplikat dihapus dari RAW.")

# 4. Fungsi pembentuk prompt (satu arah tetap)
def make_prompt_id2bugis(indo, bugis):
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "Anda adalah penerjemah profesional. Terjemahkan teks Bahasa Indonesia ke Bahasa Bugis dengan akurat."},
            {"role": "user", "content": f"Teks Indonesia: {indo}"},
            {"role": "assistant", "content": f"Terjemahan Bugis: {bugis}"}
        ],
        tokenize=False, add_generation_prompt=False
    )

def make_prompt_bugis2id(bugis, indo):
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "Anda adalah penerjemah profesional. Terjemahkan teks Bahasa Bugis ke Bahasa Indonesia dengan akurat."},
            {"role": "user", "content": f"Teks Bugis: {bugis}"},
            {"role": "assistant", "content": f"Terjemahan Indonesia: {indo}"}
        ],
        tokenize=False, add_generation_prompt=False
    )

# 5. Pisahkan: Bangun dataset per arah, lalu gabung
def add_id2bugis_prompt(example):
    return {"formatted_prompt": make_prompt_id2bugis(example["original_clean"], example["translated_clean"])}

def add_bugis2id_prompt(example):
    return {"formatted_prompt": make_prompt_bugis2id(example["translated_clean"], example["original_clean"])}

dataset_id2bugis = raw_clean.map(add_id2bugis_prompt)
dataset_bugis2id = raw_clean.map(add_bugis2id_prompt)

# Gabung dalam setiap split
from datasets import concatenate_datasets
processed_dataset = {}
for split in raw_clean.keys():
    processed_dataset[split] = concatenate_datasets([dataset_id2bugis[split], dataset_bugis2id[split]])
    print(f"Split {split}: {len(processed_dataset[split])} sampel (setelah gabung dua arah)")

jml_akhir = sum(len(processed_dataset[s]) for s in processed_dataset)
print(f"Total data siap latih: {jml_akhir}")

Cell 5: [c] Pembagian Data (Data Splitting)

In [ ]:
print("Tahap c: Pembagian Data (Data Splitting)...")

# Gabungkan semua split
combined_dataset = concatenate_datasets([
    processed_dataset['train'],
    processed_dataset['validation'],
    processed_dataset['test']
])

# === DEDUPLIKASI GLOBAL BERDASARKAN PROMPT ===
print("Melakukan deduplikasi global (berdasarkan formatted_prompt)...")
import pandas as pd
df = combined_dataset.to_pandas()
sebelum = len(df)
df.drop_duplicates(subset=['formatted_prompt'], inplace=True)  # 👈 kunci
sesudah = len(df)
print(f"Duplikat lintas split dihapus: {sebelum - sesudah}")

from datasets import Dataset, DatasetDict
combined_dataset = Dataset.from_pandas(df, preserve_index=False)

# === FILTER DATA DENGAN PANJANG TOKEN > 512 ===
print("Menyaring data dengan panjang token > 512...")
def token_length_ok(example):
    return len(tokenizer.encode(example['formatted_prompt'])) <= 512

sebelum_filter = len(combined_dataset)
combined_dataset = combined_dataset.filter(token_length_ok)
sesudah_filter = len(combined_dataset)
print(f"Sampel dibuang karena >512 token: {sebelum_filter - sesudah_filter}")

test_size_abs = 20000  
val_size_abs = 6000    

split_1 = combined_dataset.train_test_split(test_size=test_size_abs, seed=42)
temp_train_val = split_1['train']
test_set = split_1['test']

split_2 = temp_train_val.train_test_split(test_size=val_size_abs, seed=42)
train_set = split_2['train']
val_set = split_2['test']

final_dataset = DatasetDict({
    'train': train_set,
    'validation': val_set,
    'test': test_set
})

print("\n--- INFO TABEL 1: DATA SPLITTING ---")
print(f"Train : {len(train_set)} baris")
print(f"Val   : {len(val_set)} baris")
print(f"Test  : {len(test_set)} baris")

final_dataset.save_to_disk("./dataset-split-final")
print("Dataset hasil split disimpan di ./dataset-split-final")

Cell 5.5: Tokenisasi Aman

In [ ]:
print("Tokenisasi dataset secara aman (tanpa multiprocessing)...")

import os
from datasets import DatasetDict
from transformers import AutoTokenizer

# ===== Pastikan tokenizer tersedia =====
try:
    # Apakah tokenizer sudah ada di namespace?
    tokenizer
except NameError:
    print("Tokenizer belum dimuat. Memuat tokenizer dari Cell 4...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})
    tokenizer.padding_side = "right"
    # Terapkan chat template yang sama (tanpa tanggal)
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{% if message['role'] == 'system' %}"
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>"
        "{% elif message['role'] == 'user' %}"
        "<|start_header_id|>user<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>"
        "{% elif message['role'] == 'assistant' %}"
        "<|start_header_id|>assistant<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>"
        "{% endif %}"
        "{% endfor %}"
    )
    print("✅ Tokenizer dimuat ulang.")
else:
    print("✅ Tokenizer sudah tersedia.")

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def tokenize_function(examples):
    texts = examples["formatted_prompt"]
    return tokenizer(
        texts,
        padding=False,
        truncation=True,
        max_length=512
    )

# Muat final_dataset dari disk
final_dataset = DatasetDict.load_from_disk("./dataset-split-final")

# Tokenisasi dengan single process
tokenized_dataset = final_dataset.map(
    tokenize_function,
    batched=True,
    num_proc=1,
    load_from_cache_file=False,
    keep_in_memory=True,
    desc="Tokenizing"
)

tokenized_dataset.save_to_disk("./tokenized-dataset-final")
print("✅ Tokenized dataset disimpan di ./tokenized-dataset-final")

Cell 6: [d] Konfigurasi Model & QLoRA

In [ ]:
print("Tahap d: Konfigurasi Model...")

start_time = time.time()
gc.collect()
torch.cuda.empty_cache()

# 1. Muat model 4-bit dengan Unsloth (abaikan tokenizer bawaan)
model, _ = FastLanguageModel.from_pretrained(
    model_name=model_name,          # sama dengan Cell 4
    max_seq_length=512,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

# 2. Pastikan padding token sama dengan tokenizer di Cell 4
model.config.pad_token_id = tokenizer.pad_token_id

# 3. Siapkan PEFT dengan Unsloth (gradient checkpointing khusus)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,                      # 0 untuk kecepatan Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth", # hemat VRAM
    random_state=42,
    use_rslora=False,
)

end_time = time.time()
waktu_load = (end_time - start_time) / 60

print("Parameter yang dapat dilatih:")
model.print_trainable_parameters()
print(f"\nWaktu loading model: {waktu_load:.2f} menit")

Cell 7: [e] Pelatihan Model (Fine-Tuning)

In [ ]:
print("Tahap e: Pelatihan Model (Fine-Tuning)...")

import os, warnings, gc, time, glob
from datasets import DatasetDict

# Nonaktifkan multiprocessing (opsional, untuk aman)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# Muat dataset tokenized (sudah berupa token ids)
tokenized_dataset = DatasetDict.load_from_disk("./tokenized-dataset-final")
print("✅ Dataset tokenized dimuat.")

class LogHistoryCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            step = state.global_step
            train_loss = f"{logs.get('loss'):.4f}" if logs.get("loss") is not None else "-"
            val_loss = f"{logs.get('eval_loss'):.4f}" if logs.get("eval_loss") is not None else "-"
            lr = f"{logs.get('learning_rate'):.2e}" if logs.get("learning_rate") is not None else "-"
            log_type = "EVAL" if "eval_loss" in logs else "TRAIN"
            print(f"[{log_type}] Step: {step} | Train Loss: {train_loss} | Val Loss: {val_loss} | LR: {lr}")

training_arguments = SFTConfig(
    output_dir="./llama3-bugis-id-results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    optim="adamw_8bit",
    save_steps=500,
    logging_steps=100,
    learning_rate=1e-4,
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=500,
    gradient_checkpointing=False,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_only_model=False,

    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    dataloader_drop_last=True,

    # Tidak pakai dataset_text_field karena dataset sudah tokenized
    max_length=512,
    packing=True,
    remove_unused_columns=False,   # penting: jangan hapus kolom input_ids, attention_mask, dll
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset['train'],      # sudah berisi input_ids & attention_mask
    eval_dataset=tokenized_dataset['validation'],  # idem
    tokenizer=tokenizer,
    args=training_arguments,
    callbacks=[LogHistoryCallback()]
)

# Di sini kita langsung jalan kan training
checkpoint_dir = "./llama3-bugis-id-results"
existing_ckpts = sorted(glob.glob(f"{checkpoint_dir}/checkpoint-*"))
resume_from_checkpoint = existing_ckpts[-1] if existing_ckpts else None
if resume_from_checkpoint:
    print(f"🔄 Melanjutkan dari checkpoint: {resume_from_checkpoint}")
else:
    print("🆕 Tidak ada checkpoint, mulai dari awal.")

gc.collect()
torch.cuda.empty_cache()

start_time = time.time()
trainer.train(resume_from_checkpoint=resume_from_checkpoint)
end_time = time.time()
waktu_train = (end_time - start_time) / 3600
print(f"\nWaktu Training Total: {int(waktu_train)} jam {int((waktu_train % 1) * 60)} menit")

Cell 8: Menyimpan Model dan Mengosongkan Memori

In [ ]:
print("Menyimpan hasil pelatihan...")
trainer.model.save_pretrained("./llama3-bugis-id-adapter")
tokenizer.save_pretrained("./llama3-bugis-id-adapter")

print("Membersihkan memori GPU untuk persiapan Evaluasi (Inference)...")
# Menghapus objek model dari RAM dan memaksakan garbage collector
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

print_memory_usage("Setelah Memori Dibersihkan")

Cell 9: [f] Pengujian dan Evaluasi (Inference)

In [ ]:
print("Tahap f: Load Model untuk Inference & Evaluasi...")
start_time = time.time()

# === Pastikan dataset tersedia (tahan mati listrik) ===
if 'final_dataset' not in globals():
    if os.path.exists("./dataset-split-final"):
        print("Memuat dataset dari ./dataset-split-final ...")
        final_dataset = DatasetDict.load_from_disk("./dataset-split-final")
    else:
        raise FileNotFoundError(
            "Dataset tidak ditemukan. Jalankan Cell 3, 4, 5 terlebih dahulu."
        )
else:
    print("final_dataset sudah tersedia di memori.")

# === Load ulang base model dan adapter ===
if not os.path.exists("./llama3-bugis-id-adapter"):
    raise FileNotFoundError(
        "Adapter tidak ditemukan di './llama3-bugis-id-adapter'. "
        "Pastikan Cell 8 sudah dijalankan."
    )

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

model_eval = PeftModel.from_pretrained(base_model, "./llama3-bugis-id-adapter")
model_eval.eval()

# === Siapkan metrik BLEU ===
bleu = evaluate.load("sacrebleu")

# === Fungsi inferensi (perbaikan: max_new_tokens 128 agar lebih aman) ===
def generate_translation(bugis_text):
    prompt = f"Terjemahkan teks bahasa Bugis berikut ke bahasa Indonesia.\n\nTeks Bugis: {bugis_text}\n\nTerjemahan:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.inference_mode():
        outputs = model_eval.generate(
            **inputs, 
            max_new_tokens=128,       # cukup untuk kalimat pendek-sedang
            temperature=0.1, 
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    translation = response.split("Terjemahan:")[-1].strip()
    return translation

# === Pilih 100 sampel acak untuk evaluasi (kualitatif & kuantitatif) ===
import random
random.seed(42)  # agar reproduceable
all_test_indices = list(range(len(final_dataset['test'])))
sample_indices = random.sample(all_test_indices, min(100, len(final_dataset['test'])))
test_samples = final_dataset['test'].select(sample_indices)

predictions = []
references = []

print("Membuat prediksi terjemahan mesin...")
for item in test_samples:
    bug_text = item['original']            
    id_reference = item['translated']      
    pred = generate_translation(bug_text)
    predictions.append(pred)
    references.append([id_reference])

# === Simpan hasil untuk Lembar Observasi Kualitatif ===
import pandas as pd
rows = []
for i, (ref_list, pred) in enumerate(zip(references, predictions)):
    rows.append({
        "No": i + 1,
        "Teks Sumber": test_samples[i]['original'],
        "Referensi": ref_list[0],  # sudah dalam list
        "Prediksi": pred,
        "Analisis": "",            # diisi manual: Tepat / Kurang Tepat / Salah
        "Tipe Kesalahan": ""       # diisi manual: Leksikal / Gramatikal (jika bukan Tepat)
    })
df_obs = pd.DataFrame(rows)
df_obs.to_excel("lembar_observasi_kualitatif.xlsx", index=False)
print("Lembar observasi disimpan di 'lembar_observasi_kualitatif.xlsx' (siap dianalisis manual).")

# === Hitung BLEU Score (Tabel 3) ===
print("\n--- Output Tabel 3: Skenario Pengujian (Fine-Tuned Epoch 3) ---")
for i in range(1, 5):
    res = bleu.compute(predictions=predictions, references=references, max_ngram_order=i)
    print(f"BLEU-{i}: {res['score']:.2f}")

res_total = bleu.compute(predictions=predictions, references=references)
print(f"Rata-rata (Cumulative BLEU): {res_total['score']:.2f}")

# === Waktu & Memori ===
end_time = time.time()
waktu_eval = (end_time - start_time) / 60
print("\n--- Output Tabel 4: Aktivitas Inference ---")
print(f"Waktu (j:m) Inference: 0 jam {int(waktu_eval)} menit")
print_memory_usage("Setelah Inference Selesai")